<a href="https://colab.research.google.com/github/promckkon/MK2DimCNN/blob/main/AHSPSO%20with%203B%20Cat%20in%20CW%20Data%20260514.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:

# =========================================================
# Cell 1: 環境配置與套件載入 (Environment & Setup)
# =========================================================
import os
import warnings
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
warnings.filterwarnings('ignore') # 隱藏 Pandas FutureWarning 等

# 自動檢查並修復 CatBoost 與 NumPy 版本衝突
try:
    import catboost
    import numpy as np
    if catboost.__version__ != '1.2.7' or np.__version__ >= '2.0.0':
        raise ImportError
except:
    print("正在安裝穩定版環境 (NumPy 1.26.4 & CatBoost 1.2.7)...")
    get_ipython().system('pip install -q "numpy==1.26.4" "catboost==1.2.7" "scipy" "scikit-learn" "seaborn" "matplotlib"')
    print("--- 安裝完成！請點擊上方選單「執行階段」->「重新啟動執行階段」，然後再次執行此單元格 ---")

import scipy.io
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models, backend as K
from tensorflow.keras.utils import to_categorical
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split, StratifiedShuffleSplit
from sklearn.metrics import precision_recall_fscore_support, accuracy_score

from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, mutual_info_score
from sklearn.manifold import TSNE
from sklearn.utils import resample
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import skew, kurtosis, entropy

import datetime
import random
import itertools # 🌟 [修改點 3] 加入 itertools，用來跑 6x6 矩陣的排列組合

print("✅ 所有套件與高級特徵計算工具載入完成，準備就緒！")

✅ 所有套件與高級特徵計算工具載入完成，準備就緒！


In [3]:
# =========================================================
# Master Cell: 全自動端到端靜默管線 (精簡套件版 | 9特徵 -> MK-DCNN -> AHS-PSO -> 結算)
# =========================================================
import os
import time
import tracemalloc
import numpy as np
import pandas as pd
import scipy.io
from scipy.stats import skew, kurtosis
from sklearn.utils import resample
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.manifold import TSNE
from sklearn.model_selection import train_test_split, StratifiedShuffleSplit
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report
import tensorflow as tf
from tensorflow.keras import layers, models, backend as K
from tensorflow.keras.utils import to_categorical
from catboost import Pool, CatBoostClassifier

print("🚀 啟動全自動靜默訓練管線 (End-to-End Pipeline)...")
print("   [系統提示] 執行期間將關閉圖表與中途報表輸出，系統全速運算中，請耐心等候...")

# ==========================================
# ⏱️ 啟動全系統【端到端】資源監控
# ==========================================
tracemalloc.start()
global_start_time = time.time()

# ---------------------------------------------------------
# [Phase 1] 資料讀取、特徵萃取與平衡
# ---------------------------------------------------------
MAT_FOLDER_PATH = "/content/drive/MyDrive/CWRU_with_NOISE/CWRU_2_NOISE_0"
df_list = []

for root, dirs, files in os.walk(MAT_FOLDER_PATH, topdown=False):
    for file_name in files:
        if file_name.endswith('.mat'):
            path = os.path.join(root, file_name)
            mat = scipy.io.loadmat(path)
            key_name = list(mat.keys())[3]
            df_temp = pd.DataFrame({'DE_data': np.ravel(mat.get(key_name)), 'fault': file_name[:-4]})
            df_list.append(df_temp)

df = pd.concat(df_list, ignore_index=True)

TARGET_ROWS = 1800
fault_types = df['fault'].unique()
win_len = int((len(df) / len(fault_types)) / (TARGET_ROWS / len(fault_types)))
stride = int(win_len * 0.4)

X_raw_list, stats_list, y_list = [], [], []

for f in fault_types:
    fault_data = df[df['fault'] == f]['DE_data'].values.astype(float)
    num_windows = (len(fault_data) - win_len) // stride + 1
    for i in range(num_windows):
        window = fault_data[i*stride : i*stride + win_len]
        X_raw_list.append(window.reshape(-1, 1))
        y_list.append(f)

        rms = np.sqrt(np.mean(np.square(window)))
        mean_abs = np.mean(np.abs(window))

        # 9 個基礎統計特徵
        stats_list.append([
            np.mean(window), np.std(window), rms, np.max(window), np.min(window),
            skew(window), kurtosis(window),
            rms / mean_abs if mean_abs != 0 else 0,
            np.max(window) / rms if rms != 0 else 0
        ])

temp_df = pd.DataFrame({'X_raw': X_raw_list, 'stats': stats_list, 'fault': y_list})
TARGET_BALANCED_ROWS = 1570
samples_per_class = TARGET_BALANCED_ROWS // len(fault_types)

resampled_data = [resample(temp_df[temp_df['fault'] == f],
                           replace=(len(temp_df[temp_df['fault'] == f]) < samples_per_class),
                           n_samples=samples_per_class, random_state=42) for f in fault_types]

balanced_df = pd.concat(resampled_data).sample(frac=1, random_state=42).reset_index(drop=True)

X_raw_balanced = np.stack(balanced_df['X_raw'].values)
X_stat_balanced = np.stack(balanced_df['stats'].values)
y_label = balanced_df['fault'].values

encoder = LabelEncoder()
y_encoded = encoder.fit_transform(y_label)
y_categorical = to_categorical(y_encoded)

# ---------------------------------------------------------
# [Phase 2] MK-DCNN 訓練與特徵融合 (Fusion)
# ---------------------------------------------------------
def custom_loss(y_true, y_pred):
    loss = K.categorical_crossentropy(y_true, y_pred)
    if K.int_shape(y_pred)[-1] > 1:
        physics_term = tf.reduce_mean(tf.square(y_pred[:, 1:] - y_pred[:, :-1]))
    else:
        physics_term = 0.0
    return loss + 0.01 * physics_term

inputs = layers.Input(shape=(X_raw_balanced.shape[1], 1))
flat1 = layers.Flatten()(layers.MaxPooling1D(20)(layers.Dropout(0.5)(layers.Conv1D(64, 200, activation='relu')(inputs))))
flat2 = layers.Flatten()(layers.MaxPooling1D(10)(layers.Dropout(0.5)(layers.Conv1D(64, 100, activation='relu')(inputs))))
flat3 = layers.Flatten()(layers.MaxPooling1D(5)(layers.Dropout(0.5)(layers.Conv1D(64, 50, activation='relu')(inputs))))

merged = layers.concatenate([flat1, flat2, flat3])
dense_layer = layers.Dense(100, activation='relu', name='deep_feature')(merged)
outputs = layers.Dense(len(encoder.classes_), activation='softmax')(dense_layer)

cnn_model = models.Model(inputs=inputs, outputs=outputs)
cnn_model.compile(optimizer='adam', loss=custom_loss, metrics=['accuracy'])
cnn_model.fit(X_raw_balanced, y_categorical, batch_size=100, epochs=20, verbose=0)

dummy_cnn = models.Model(inputs=cnn_model.input, outputs=cnn_model.get_layer('deep_feature').output)
X_deep = dummy_cnn.predict(X_raw_balanced, verbose=0)

tsne = TSNE(n_components=2, perplexity=40, random_state=42)
X_stat_tsne = tsne.fit_transform(StandardScaler().fit_transform(X_stat_balanced))
X_deep_tsne = tsne.fit_transform(X_deep)

# 水平融合: 2維統計 + 2維深度 = 4維融合特徵
X_fused_4d = np.concatenate((X_stat_tsne, X_deep_tsne), axis=1)

# ---------------------------------------------------------
# [Phase 3] AHS-PSO 最佳化與最終 CatBoost 模型部署
# ---------------------------------------------------------
class AHS_PSO:
    def __init__(self, X_tr, y_tr, X_va, y_va, search_space, n_part=10, max_iter=15, seed=42):
        self.X_tr, self.y_tr, self.X_va, self.y_va = X_tr, y_tr, X_va, y_va
        self.search_space = search_space
        self.keys = list(search_space.keys())
        self.dim, self.n_part, self.max_iter = len(self.keys), n_part, max_iter
        self.rng = np.random.default_rng(seed)

        self.switch_threshold, self.w_min, self.w_max = 5, 0.5, 0.9
        self.c1, self.c2, self.mutation_prob, self.restart_prob = 0.8, 0.6, 0.15, 0.08
        self.target_acc, self.H, self.b = 0.999, 1.0, 0.9
        self.base_gamma = 0.25
        self.eval_count = 0
        self.stagnation_limit = 3

    def project_to_space(self, key, value):
        choices = np.array(self.search_space[key], dtype=float)
        v = self.search_space[key][int(np.argmin(np.abs(choices - float(value))))]
        return int(v) if isinstance(self.search_space[key][0], int) else float(v)

    def clip_cast(self, p):
        return {
            "iterations": int(self.project_to_space("iterations", p["iterations"])),
            "depth": int(self.project_to_space("depth", p["depth"])),
            "learning_rate": self.project_to_space("learning_rate", p["learning_rate"]),
            "l2_leaf_reg": self.project_to_space("l2_leaf_reg", p["l2_leaf_reg"]),
            "bagging_temperature": self.project_to_space("bagging_temperature", p["bagging_temperature"]),
            "random_strength": self.project_to_space("random_strength", p["random_strength"])
        }

    def obj_fast(self, p):
        self.eval_count += 1
        model = CatBoostClassifier(**self.clip_cast(p), loss_function="MultiClass", eval_metric="Accuracy",
                                   od_type="Iter", od_wait=20, verbose=False, random_seed=42, thread_count=-1)
        model.fit(self.X_tr, self.y_tr, eval_set=(self.X_va, self.y_va), use_best_model=True)
        return float(accuracy_score(self.y_va, model.predict(self.X_va).reshape(-1)))

    def pt_to_vec(self, p): return np.array([float(p[k]) for k in self.keys], dtype=float)
    def vec_to_pt(self, v): return {k: float(v[i]) for i, k in enumerate(self.keys)}

    def optimize(self):
        swarm = [{k: self.rng.choice(self.search_space[k]) for k in self.keys} for _ in range(self.n_part)]
        vel = [np.zeros(self.dim, dtype=float) for _ in range(self.n_part)]
        pbest = [dict(s) for s in swarm]

        pbest_score = [self.obj_fast(s) for s in swarm]
        gbest = dict(pbest[int(np.argmax(pbest_score))])
        gbest_score = float(np.max(pbest_score))

        topology = "gbest"
        stagnation_counter = 0
        current_gamma = self.base_gamma

        for it in range(1, self.max_iter + 1):
            if gbest_score >= self.target_acc:
                break

            w = self.w_max - ((self.w_max - self.w_min) * it) / (1.0 + np.exp(-10.0 * self.b * ((2.0 * it) / (self.H * self.max_iter) - 1.0)))
            if it % self.switch_threshold == 0: topology = "lbest" if topology == "gbest" else "gbest"

            improved_this_iter = False
            for i in range(self.n_part):
                if topology == "gbest": g_vec = self.pt_to_vec(gbest)
                else: g_vec = self.pt_to_vec(pbest[self.rng.choice(self.n_part, min(5, self.n_part), replace=False)[np.argmax([pbest_score[j] for j in self.rng.choice(self.n_part, min(5, self.n_part), replace=False)])]])

                vel[i] = w * vel[i] + self.c1 * self.rng.random(self.dim) * (self.pt_to_vec(pbest[i]) - self.pt_to_vec(swarm[i])) + self.c2 * self.rng.random(self.dim) * (g_vec - self.pt_to_vec(swarm[i]))
                swarm[i] = self.clip_cast(self.vec_to_pt(self.pt_to_vec(swarm[i]) + vel[i]))

                if self.rng.random() < self.mutation_prob:
                    mut = dict(swarm[i])
                    for k in self.keys:
                        mut[k] = self.project_to_space(k, float(mut[k]) * (1.0 + current_gamma * np.tan(np.pi * (self.rng.random() - 0.5))))
                    swarm[i] = self.clip_cast(mut)

                if self.rng.random() < self.restart_prob:
                    swarm[i] = {k: self.rng.choice(self.search_space[k]) for k in self.keys}

                s = self.obj_fast(swarm[i])
                if s > pbest_score[i]:
                    pbest[i], pbest_score[i] = dict(swarm[i]), float(s)

            best_i = int(np.argmax(pbest_score))
            if pbest_score[best_i] > gbest_score:
                gbest, gbest_score = dict(pbest[best_i]), float(pbest_score[best_i])
                improved_this_iter = True

            if improved_this_iter:
                stagnation_counter = 0
                current_gamma = self.base_gamma
            else:
                stagnation_counter += 1
                if stagnation_counter >= self.stagnation_limit:
                    current_gamma = min(1.0, current_gamma * 1.5)

        return self.clip_cast(gbest), gbest_score, self.eval_count

X_train, X_test, y_train, y_test = train_test_split(X_fused_4d, y_encoded, test_size=0.2, random_state=42)
sss = StratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
tr_idx, va_idx = next(sss.split(X_train, y_train))
X_tr, y_tr, X_va, y_va = X_train[tr_idx], y_train[tr_idx], X_train[va_idx], y_train[va_idx]

search_space = {
    "iterations": list(range(50, 201, 25)), "depth": list(range(2, 9)),
    "learning_rate": [0.03, 0.05, 0.08, 0.1], "l2_leaf_reg": [1.0, 3.0, 5.0],
    "bagging_temperature": [0.5, 1.0, 1.5], "random_strength": [0.5, 1.0, 1.5]
}

optimizer = AHS_PSO(X_tr, y_tr, X_va, y_va, search_space, n_part=10, max_iter=15)
best_params, best_acc, total_evals = optimizer.optimize()

final_model = CatBoostClassifier(**best_params, loss_function="MultiClass", eval_metric="Accuracy",
                                 od_type="Iter", od_wait=40, verbose=0, random_seed=42, thread_count=-1)
final_model.fit(Pool(X_train, y_train), eval_set=Pool(X_test, y_test), use_best_model=True)

y_pred_test = final_model.predict(X_test).reshape(-1)
test_acc = accuracy_score(y_test, y_pred_test)
precision, recall, f1, _ = precision_recall_fscore_support(y_test, y_pred_test, average='macro', zero_division=0)

# ==========================================
# ⏱️ 結束全系統【端到端】資源監控與結算輸出
# ==========================================
global_end_time = time.time()
_, global_total_mem = tracemalloc.get_traced_memory()
tracemalloc.stop()

global_time_sec = global_end_time - global_start_time
global_mem_mb = global_total_mem / (1024 * 1024)

print("✅ 所有管線任務已成功執行完畢！\n")
print("="*60)
print(" 🏆 【端到端】系統整體計算成本與效能報告 (End-to-End Report)")
print("="*60)
print(f"⏱️ 系統總耗時 (Total Time)  : {global_time_sec:.2f} 秒 ({global_time_sec/60:.2f} 分鐘)")
print(f"💾 系統總記憶體 (Total Mem) : {global_mem_mb:.2f} MB")
print(f"⚙️ 尋優評估次數 (PSO Evals): {total_evals} 次")
print(f"🎯 最佳超參數組合 : {best_params}")
print("="*60)
print("\n 🚀 最終融合模型測試集量化評估指標")
print("="*60)
print(f"✅ Test Accuracy (整體準確率) : {test_acc * 100:.2f}%")
print(f"✅ Average Precision (平均精確率): {precision * 100:.2f}%")
print(f"✅ Average Recall (平均召回率)   : {recall * 100:.2f}%")
print(f"✅ Average F1-Score (平均 F1)    : {f1 * 100:.2f}%")
print("="*60)

🚀 啟動全自動靜默訓練管線 (End-to-End Pipeline)...
   [系統提示] 執行期間將關閉圖表與中途報表輸出，系統全速運算中，請耐心等候...
✅ 所有管線任務已成功執行完畢！

 🏆 【端到端】系統整體計算成本與效能報告 (End-to-End Report)
⏱️ 系統總耗時 (Total Time)  : 223.82 秒 (3.73 分鐘)
💾 系統總記憶體 (Total Mem) : 266.06 MB
⚙️ 尋優評估次數 (PSO Evals): 160 次
🎯 最佳超參數組合 : {'iterations': 175, 'depth': 7, 'learning_rate': 0.08, 'l2_leaf_reg': 5.0, 'bagging_temperature': 1.0, 'random_strength': 0.5}

 🚀 最終融合模型測試集量化評估指標
✅ Test Accuracy (整體準確率) : 100.00%
✅ Average Precision (平均精確率): 100.00%
✅ Average Recall (平均召回率)   : 100.00%
✅ Average F1-Score (平均 F1)    : 100.00%


In [6]:
# =========================================================
# Master Cell: 全自動極速靜默管線 (9特徵版 + PCA極速降維 + PSO 絕對提早停止)
# =========================================================
import os
import time
import tracemalloc
import numpy as np
import pandas as pd
import scipy.io
from scipy.stats import skew, kurtosis
from sklearn.utils import resample
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.decomposition import PCA  # 🚀 [加速點 1] 替換 t-SNE 為極速 PCA
from sklearn.model_selection import train_test_split, StratifiedShuffleSplit
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report
import tensorflow as tf
from tensorflow.keras import layers, models, backend as K
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.callbacks import EarlyStopping # 🚀 [加速點 2] CNN 提早停止
from catboost import Pool, CatBoostClassifier

print("🚀 啟動全自動極速靜默訓練管線 (Ultra-Fast End-to-End Pipeline)...")
print("   [系統提示] 啟動 PCA 極速特徵降維與 100% 強制提早停止機制...")

# ==========================================
# ⏱️ 啟動全系統資源監控
# ==========================================
tracemalloc.start()
global_start_time = time.time()

# ---------------------------------------------------------
# [Phase 1] 資料讀取、特徵萃取與平衡
# ---------------------------------------------------------
MAT_FOLDER_PATH = "/content/drive/MyDrive/CWRU_with_NOISE/CWRU_2_NOISE_0"
df_list = []

for root, dirs, files in os.walk(MAT_FOLDER_PATH, topdown=False):
    for file_name in files:
        if file_name.endswith('.mat'):
            path = os.path.join(root, file_name)
            mat = scipy.io.loadmat(path)
            key_name = list(mat.keys())[3]
            df_temp = pd.DataFrame({'DE_data': np.ravel(mat.get(key_name)), 'fault': file_name[:-4]})
            df_list.append(df_temp)

if not df_list: raise FileNotFoundError(f"🚨 找不到 .mat 檔案，請檢查路徑：{MAT_FOLDER_PATH}")
df = pd.concat(df_list, ignore_index=True)

TARGET_ROWS = 1600
fault_types = df['fault'].unique()
win_len = int((len(df) / len(fault_types)) / (TARGET_ROWS / len(fault_types)))
stride = int(win_len * 0.4)

X_raw_list, stats_list, y_list = [], [], []

for f in fault_types:
    fault_data = df[df['fault'] == f]['DE_data'].values.astype(float)
    num_windows = (len(fault_data) - win_len) // stride + 1
    for i in range(num_windows):
        window = fault_data[i*stride : i*stride + win_len]
        X_raw_list.append(window.reshape(-1, 1))
        y_list.append(f)

        rms = np.sqrt(np.mean(np.square(window)))
        mean_abs = np.mean(np.abs(window))

        # 9 個基礎統計特徵
        stats_list.append([
            np.mean(window), np.std(window), rms, np.max(window), np.min(window),
            skew(window), kurtosis(window),
            rms / mean_abs if mean_abs != 0 else 0,
            np.max(window) / rms if rms != 0 else 0
        ])

temp_df = pd.DataFrame({'X_raw': X_raw_list, 'stats': stats_list, 'fault': y_list})
TARGET_BALANCED_ROWS = 1570
samples_per_class = TARGET_BALANCED_ROWS // len(fault_types)

resampled_data = [resample(temp_df[temp_df['fault'] == f],
                           replace=(len(temp_df[temp_df['fault'] == f]) < samples_per_class),
                           n_samples=samples_per_class, random_state=42) for f in fault_types]

balanced_df = pd.concat(resampled_data).sample(frac=1, random_state=42).reset_index(drop=True)

X_raw_balanced = np.stack(balanced_df['X_raw'].values)
X_stat_balanced = np.stack(balanced_df['stats'].values)
y_label = balanced_df['fault'].values

encoder = LabelEncoder()
y_encoded = encoder.fit_transform(y_label)
y_categorical = to_categorical(y_encoded)

# ---------------------------------------------------------
# [Phase 2] MK-DCNN 訓練與特徵融合 (PCA 極速版)
# ---------------------------------------------------------
def custom_loss(y_true, y_pred):
    loss = K.categorical_crossentropy(y_true, y_pred)
    if K.int_shape(y_pred)[-1] > 1:
        physics_term = tf.reduce_mean(tf.square(y_pred[:, 1:] - y_pred[:, :-1]))
    else:
        physics_term = 0.0
    return loss + 0.01 * physics_term

inputs = layers.Input(shape=(X_raw_balanced.shape[1], 1))
flat1 = layers.Flatten()(layers.MaxPooling1D(20)(layers.Dropout(0.5)(layers.Conv1D(64, 200, activation='relu')(inputs))))
flat2 = layers.Flatten()(layers.MaxPooling1D(10)(layers.Dropout(0.5)(layers.Conv1D(64, 100, activation='relu')(inputs))))
flat3 = layers.Flatten()(layers.MaxPooling1D(5)(layers.Dropout(0.5)(layers.Conv1D(64, 50, activation='relu')(inputs))))

merged = layers.concatenate([flat1, flat2, flat3])
dense_layer = layers.Dense(100, activation='relu', name='deep_feature')(merged)
outputs = layers.Dense(len(encoder.classes_), activation='softmax')(dense_layer)

cnn_model = models.Model(inputs=inputs, outputs=outputs)
cnn_model.compile(optimizer='adam', loss=custom_loss, metrics=['accuracy'])

# 🚀 提早停止：如果 loss 停滯 3 個 epoch 就不跑了
early_stopper = EarlyStopping(monitor='loss', patience=3, restore_best_weights=True)
cnn_model.fit(X_raw_balanced, y_categorical, batch_size=100, epochs=20, verbose=0, callbacks=[early_stopper])

dummy_cnn = models.Model(inputs=cnn_model.input, outputs=cnn_model.get_layer('deep_feature').output)
X_deep = dummy_cnn.predict(X_raw_balanced, verbose=0)

# 🚀 使用 PCA 取代 t-SNE (運算時間從幾十秒降至 0.01 秒)
pca = PCA(n_components=2, random_state=42)
X_stat_pca = pca.fit_transform(StandardScaler().fit_transform(X_stat_balanced))
X_deep_pca = pca.fit_transform(X_deep)

X_fused_4d = np.concatenate((X_stat_pca, X_deep_pca), axis=1)

# ---------------------------------------------------------
# [Phase 3] AHS-PSO 最佳化與最終模型部署
# ---------------------------------------------------------
class AHS_PSO:
    def __init__(self, X_tr, y_tr, X_va, y_va, search_space, n_part=10, max_iter=15, seed=42):
        self.X_tr, self.y_tr, self.X_va, self.y_va = X_tr, y_tr, X_va, y_va
        self.search_space = search_space
        self.keys = list(search_space.keys())
        self.dim, self.n_part, self.max_iter = len(self.keys), n_part, max_iter
        self.rng = np.random.default_rng(seed)

        self.switch_threshold, self.w_min, self.w_max = 5, 0.5, 0.9
        self.c1, self.c2, self.mutation_prob, self.restart_prob = 0.8, 0.6, 0.15, 0.08
        self.target_acc, self.H, self.b = 0.999, 1.0, 0.9
        self.base_gamma = 0.25
        self.eval_count = 0
        self.stagnation_limit = 3

    def project_to_space(self, key, value):
        choices = np.array(self.search_space[key], dtype=float)
        v = self.search_space[key][int(np.argmin(np.abs(choices - float(value))))]
        return int(v) if isinstance(self.search_space[key][0], int) else float(v)

    def clip_cast(self, p):
        return {
            "iterations": int(self.project_to_space("iterations", p["iterations"])),
            "depth": int(self.project_to_space("depth", p["depth"])),
            "learning_rate": self.project_to_space("learning_rate", p["learning_rate"]),
            "l2_leaf_reg": self.project_to_space("l2_leaf_reg", p["l2_leaf_reg"]),
            "bagging_temperature": self.project_to_space("bagging_temperature", p["bagging_temperature"]),
            "random_strength": self.project_to_space("random_strength", p["random_strength"])
        }

    def obj_fast(self, p):
        self.eval_count += 1
        model = CatBoostClassifier(**self.clip_cast(p), loss_function="MultiClass", eval_metric="Accuracy",
                                   od_type="Iter", od_wait=10, verbose=False, random_seed=42, thread_count=-1) # od_wait 降為 10 加速
        model.fit(self.X_tr, self.y_tr, eval_set=(self.X_va, self.y_va), use_best_model=True)
        return float(accuracy_score(self.y_va, model.predict(self.X_va).reshape(-1)))

    def pt_to_vec(self, p): return np.array([float(p[k]) for k in self.keys], dtype=float)
    def vec_to_pt(self, v): return {k: float(v[i]) for i, k in enumerate(self.keys)}

    def optimize(self):
        swarm = [{k: self.rng.choice(self.search_space[k]) for k in self.keys} for _ in range(self.n_part)]
        vel = [np.zeros(self.dim, dtype=float) for _ in range(self.n_part)]
        pbest = [dict(s) for s in swarm]

        pbest_score = [self.obj_fast(s) for s in swarm]
        gbest = dict(pbest[int(np.argmax(pbest_score))])
        gbest_score = float(np.max(pbest_score))

        topology = "gbest"
        stagnation_counter = 0
        current_gamma = self.base_gamma

        for it in range(1, self.max_iter + 1):
            if gbest_score >= self.target_acc: break

            w = self.w_max - ((self.w_max - self.w_min) * it) / (1.0 + np.exp(-10.0 * self.b * ((2.0 * it) / (self.H * self.max_iter) - 1.0)))
            if it % self.switch_threshold == 0: topology = "lbest" if topology == "gbest" else "gbest"

            improved_this_iter = False
            for i in range(self.n_part):
                if topology == "gbest": g_vec = self.pt_to_vec(gbest)
                else: g_vec = self.pt_to_vec(pbest[self.rng.choice(self.n_part, min(5, self.n_part), replace=False)[np.argmax([pbest_score[j] for j in self.rng.choice(self.n_part, min(5, self.n_part), replace=False)])]])

                vel[i] = w * vel[i] + self.c1 * self.rng.random(self.dim) * (self.pt_to_vec(pbest[i]) - self.pt_to_vec(swarm[i])) + self.c2 * self.rng.random(self.dim) * (g_vec - self.pt_to_vec(swarm[i]))
                swarm[i] = self.clip_cast(self.vec_to_pt(self.pt_to_vec(swarm[i]) + vel[i]))

                if self.rng.random() < self.mutation_prob:
                    mut = dict(swarm[i])
                    for k in self.keys:
                        mut[k] = self.project_to_space(k, float(mut[k]) * (1.0 + current_gamma * np.tan(np.pi * (self.rng.random() - 0.5))))
                    swarm[i] = self.clip_cast(mut)

                if self.rng.random() < self.restart_prob:
                    swarm[i] = {k: self.rng.choice(self.search_space[k]) for k in self.keys}

                s = self.obj_fast(swarm[i])
                if s > pbest_score[i]:
                    pbest[i], pbest_score[i] = dict(swarm[i]), float(s)

                # 🚀 [加速點 3] 只要這一個粒子達標 99.9%，立刻無情強制切斷所有尋優！
                if pbest_score[i] > gbest_score:
                    gbest, gbest_score = dict(pbest[i]), float(pbest_score[i])
                    improved_this_iter = True
                    if gbest_score >= self.target_acc:
                        break # 切斷內層粒子迴圈

            if gbest_score >= self.target_acc:
                break # 切斷外層迭代迴圈

            if improved_this_iter:
                stagnation_counter = 0; current_gamma = self.base_gamma
            else:
                stagnation_counter += 1
                if stagnation_counter >= self.stagnation_limit: current_gamma = min(1.0, current_gamma * 1.5)

        return self.clip_cast(gbest), gbest_score, self.eval_count

X_train, X_test, y_train, y_test = train_test_split(X_fused_4d, y_encoded, test_size=0.2, random_state=42)
sss = StratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
tr_idx, va_idx = next(sss.split(X_train, y_train))
X_tr, y_tr, X_va, y_va = X_train[tr_idx], y_train[tr_idx], X_train[va_idx], y_train[va_idx]

search_space = {
    "iterations": list(range(50, 201, 25)), "depth": list(range(2, 9)),
    "learning_rate": [0.03, 0.05, 0.08, 0.1], "l2_leaf_reg": [1.0, 3.0, 5.0],
    "bagging_temperature": [0.5, 1.0, 1.5], "random_strength": [0.5, 1.0, 1.5]
}

optimizer = AHS_PSO(X_tr, y_tr, X_va, y_va, search_space, n_part=10, max_iter=15)
best_params, best_acc, total_evals = optimizer.optimize()

final_model = CatBoostClassifier(**best_params, loss_function="MultiClass", eval_metric="Accuracy",
                                 od_type="Iter", od_wait=40, verbose=0, random_seed=42, thread_count=-1)
final_model.fit(Pool(X_train, y_train), eval_set=Pool(X_test, y_test), use_best_model=True)

y_pred_test = final_model.predict(X_test).reshape(-1)
test_acc = accuracy_score(y_test, y_pred_test)
precision, recall, f1, _ = precision_recall_fscore_support(y_test, y_pred_test, average='macro', zero_division=0)

# ==========================================
# ⏱️ 結束全系統資源監控與結算輸出
# ==========================================
global_end_time = time.time()
_, global_total_mem = tracemalloc.get_traced_memory()
tracemalloc.stop()

global_time_sec = global_end_time - global_start_time
global_mem_mb = global_total_mem / (1024 * 1024)

print("✅ 所有管線任務已極速執行完畢！\n")
print("="*60)
print(" 🏆 【極速版端到端】系統整體計算成本報告")
print("="*60)
print(f"⏱️ 系統總耗時 (Total Time)  : {global_time_sec:.2f} 秒 ({global_time_sec/60:.2f} 分鐘)")
print(f"💾 系統總記憶體 (Total Mem) : {global_mem_mb:.2f} MB")
print(f"⚙️ 尋優評估次數 (PSO Evals): {total_evals} 次 (100%強制提早截斷)")
print(f"🎯 最佳超參數組合 : {best_params}")
print("="*60)
print("\n 🚀 最終融合模型測試集量化評估指標")
print("="*60)
print(f"✅ Test Accuracy (整體準確率) : {test_acc * 100:.2f}%")
print(f"✅ Average Precision (平均精確率): {precision * 100:.2f}%")
print(f"✅ Average Recall (平均召回率)   : {recall * 100:.2f}%")
print(f"✅ Average F1-Score (平均 F1)    : {f1 * 100:.2f}%")
print("="*60)

🚀 啟動全自動極速靜默訓練管線 (Ultra-Fast End-to-End Pipeline)...
   [系統提示] 啟動 PCA 極速特徵降維與 100% 強制提早停止機制...


KeyboardInterrupt: 